# Calogero-Sutherland model in a harmonic trap (1D)

`N` particles on a line, harmonically trapped, interacting through the inverse-square
Calogero-Sutherland potential. In the code convention $\hbar^2/m = 1$ (kinetic
$=-\sum_i\partial_i^2$, **not** $-\tfrac12\partial_i^2$):

$$ H = -\sum_i \frac{\partial^2}{\partial x_i^2} + \sum_i x_i^2 + 2L(L-1)\sum_{i<j}\frac{1}{(x_i-x_j)^2} $$

**Exact ground state** (Calogero):

$$ \psi_0(x) = C \prod_{i<j}|x_i-x_j|^{L}\,\exp\!\Big(-\tfrac12\sum_i x_i^2\Big),\qquad \log|\psi_0| = L\sum_{i<j}\log|x_i-x_j| - \tfrac12\sum_i x_i^2 $$

**Exact energy** (in this $\omega=1$ convention):
$$ E_0 = N\big(1 + L(N-1)\big), \qquad E_0/N = 1 + L(N-1). $$

This is the CS analogue of the `soft_sphere_gas` notebook: a **parametrised setup** that
builds the `CalogeroSutherlandHamiltonian`, trains an ansatz, and wraps everything in
`run_cs(...)` so you can scan the coupling `L` (or `N`) and compare against the exact $E_0$.

All machinery lives in `qvarnet`:
`qvarnet.hamiltonian.continuous.CalogeroSutherlandHamiltonian`,
`qvarnet.models.analytic.CalogeroSutherlandAnalyticModel`, and `LogJastrow`.

## Imports

In [1]:
%matplotlib inline
import jax
import jax.numpy as jnp
import numpy as np
import optax
import matplotlib.pyplot as plt
import flax.linen as nn

from qvarnet.train import train
from qvarnet.config.training_setup import TrainingConfig
from qvarnet.config.coord_mode import LabCoords
from qvarnet.boundaries import NoBoundary
from qvarnet.models.compose import LogWavefunction
from qvarnet.models.mlp import MLP
from qvarnet.models.envelopes import GaussianEnvelope
from qvarnet.models.jastrow import LogJastrow
from qvarnet.models.analytic import CalogeroSutherlandAnalyticModel
from qvarnet.hamiltonian.continuous import CalogeroSutherlandHamiltonian

## 0. Parameters

`L` is the CS coupling (the exact Jastrow exponent $\lambda = L$). `L>1` is repulsive,
`L<1` attractive (and harder). The interaction strength is $g = 2L(L-1)$.

In [2]:
N      = 5          # number of particles
N_DIM  = 1          # CS is a 1-D model
L      = 0.8        # CS coupling (THE scan knob); exact lambda = L
EPS    = 1e-4       # softens the 1/(x_i-x_j)^2 singularity (small but nonzero)

DoF = N * N_DIM
E_exact_total = N * (1 + L * (N - 1))          # code convention, omega=1
E_exact_per_n = E_exact_total / N
print(f'N={N}, L={L}  ->  g = 2L(L-1) = {2*L*(L-1):.3f}')
print(f'Exact E0 = {E_exact_total:.4f}   (E0/N = {E_exact_per_n:.4f})')

N=5, L=0.8  ->  g = 2L(L-1) = -0.320
Exact E0 = 21.0000   (E0/N = 4.2000)


## 1. Ansatz and Hamiltonian

Two ready options:

- **Analytic** (`CalogeroSutherlandAnalyticModel`): the exact log-$\psi$ with a single learnable
  $\lambda$. Converges to the exact energy — the validation baseline.
- **MLP + Gaussian envelope + Jastrow**: a flexible NN ansatz (`LogWavefunction`). The `LogJastrow`
  supplies the $\lambda\sum_{i<j}\log|x_i-x_j|$ cusp that the smooth MLP cannot represent.

`CalogeroSutherlandHamiltonian(L=L, epsilon=EPS)` supplies the kinetic + trap + inverse-square
interaction (the factor-of-2 conventions are handled inside).

In [3]:
LAMBDA_INIT = 1.2   # starting guess for the Jastrow exponent; should drift toward L



class NoNetwork(nn.Module):
    """Network that contributes nothing to log|ψ|: returns zeros of shape (..., 1).

    LogWavefunction sums network + envelope + jastrow, each expected as (..., 1).
    Returning x (shape (..., N)) would broadcast log|ψ| to a vector AND inject the
    raw coordinates into the wavefunction. A constant 0 leaves only envelope+jastrow.
    """
    @nn.compact
    def __call__(self, x):
        return jnp.zeros((*x.shape[:-1], 1))

def make_model(kind='mlp_jastrow', lambda_init=LAMBDA_INIT):
    if kind == 'analytic':
        return CalogeroSutherlandAnalyticModel(lambda_init=lambda_init)
    if kind == 'mlp_jastrow':
        return LogWavefunction(
            transform=NoBoundary(),
            network=MLP(hidden=[128]),
            envelope=GaussianEnvelope(),
            jastrow=LogJastrow(n_particles=N, lambda_init=lambda_init),
        )
    if kind == 'mlp':
        return LogWavefunction(
            transform=NoBoundary(),
            network=MLP(hidden=[128]),
            envelope=GaussianEnvelope(),
        )
    if kind == 'jastrow':

        return LogWavefunction(
            transform=NoBoundary(),
            envelope=GaussianEnvelope(),
            jastrow=LogJastrow(n_particles=N, lambda_init=lambda_init),
            network=NoNetwork(),
        )
    raise ValueError(kind)

def make_hamiltonian(L, eps=EPS):
    return CalogeroSutherlandHamiltonian(L=L, epsilon=eps)

## 2. Train

Open-boundary Metropolis sampler (the trap confines the walkers). `is_update_step_size` adapts the
step to the target acceptance. `run_cs(...)` is the unit of a parameter study; it returns the
`TrainResult`, the tail-averaged energy and its error, and prints the gap to the exact $E_0$.

In [4]:
def run_cs(L, N=N, kind='mlp_jastrow', n_epochs=2000, n_chains=4096,
           lr=1e-3, lambda_init=LAMBDA_INIT, seed=0):
    '''Train the CS model at coupling L. Returns (result, E_mean, E_err).'''
    E0 = N * (1 + L * (N - 1))
    print(f'Running CS: N={N}, L={L}, model={kind}  (exact E0={E0:.4f})')
    model = make_model(kind, lambda_init)
    sampler_params = {
        'step_size':            0.5,
        'chain_length':         21,
        'thermalization_steps': 20,
        'thinning_factor':      1,
        # open boundary (no box_L): the harmonic trap confines the walkers
    }
    cfg = TrainingConfig(
        n_epochs=n_epochs, rng_seed=seed,
        warm_walkers=True, is_update_step_size=True,
        min_step=1e-5, max_step=5.0,
        checkpoint_path='./checkpoints/cs',
    )
    result = train(
        shape=(n_chains, N * N_DIM), model=model, optimizer=optax.adam(lr),
        hamiltonian=make_hamiltonian(L),
        training_config=cfg, sampler_params=sampler_params, coord_mode=LabCoords(),
    )
    e = np.array([float(s.energy) for s in result.history])
    tail = e[-max(200, n_epochs // 5):]
    E, Eerr = float(tail.mean()), float(tail.std() / np.sqrt(len(tail)))
    print(f'E = {E:.4f} +/- {Eerr:.4f}   (exact {E0:.4f}, gap {E-E0:+.4f})')
    return result, E, Eerr

result, E, Eerr = run_cs(L, kind="jastrow")
energies = np.array([float(s.energy) for s in result.history])
plt.figure(figsize=(6, 3))
plt.plot(energies, lw=0.8)
plt.axhline(E_exact_total, color='k', ls='--', lw=0.8, label='exact $E_0$')
plt.xlabel('epoch'); plt.ylabel('energy'); plt.legend()
plt.title('CS energy convergence'); plt.tight_layout(); plt.savefig('cs_energy.png', dpi=120, bbox_inches='tight'); plt.show()

Running CS: N=5, L=0.8, model=jastrow  (exact E0=21.0000)


100%|██████████| 2000/2000 [00:11<00:00, 174.18it/s, E=27.5600, sigma_E=72.8040] 


E = 30.9468 +/- 1.5995   (exact 21.0000, gap +9.9468)


/tmp/ipykernel_7030/3250477365.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title('CS energy convergence'); plt.tight_layout(); plt.show()


## 3. Scan the coupling `L`

`run_cs(L=...)` (or vary `N`) is the unit of a parameter study. The exact line
$E_0/N = 1 + L(N-1)$ is the validation target.

In [ ]:
L_scan = [0.5, 0.8, 1.0, 1.5, 2.0]
rows = []
for Lv in L_scan:
    _, Ev, errv = run_cs(Lv, n_epochs=1500, n_chains=2048)
    rows.append((Lv, Ev / N, errv / N, 1 + Lv * (N - 1)))

Ls   = [r[0] for r in rows]
plt.figure(figsize=(6, 3))
plt.errorbar(Ls, [r[1] for r in rows], yerr=[r[2] for r in rows], fmt='o', label='VMC')
plt.plot(Ls, [r[3] for r in rows], 'k--', label='exact $1+L(N-1)$')
plt.xlabel('L (coupling)'); plt.ylabel('E / N'); plt.legend()
plt.title('CS energy per particle vs coupling'); plt.tight_layout(); plt.savefig('cs_scan.png', dpi=120, bbox_inches='tight'); plt.show()

Running CS: N=5, L=0.5, model=mlp_jastrow  (exact E0=15.0000)


 44%|████▍     | 657/1500 [00:09<00:05, 161.40it/s, E=21.9862, sigma_E=9.7626]  

## Notes / next steps
- Swap `kind='analytic'` in `run_cs` for the exact single-parameter baseline (fastest convergence).
- `L<1` (attractive) is numerically harder; raise `n_chains`/`n_epochs` and consider the cusp
  condition (`TrainingConfig(cusp=CuspConfig(...))`, with `C_n=L`, `n=2`) to pin the cusp.
- Inspect the learned Jastrow $\lambda$ in `result.best_params()` — it should sit near `L`.
- Compare against `qvarnet/notebooks/scripts/cs_model_new.ipynb` for the cusp-condition study.